# Task 2: Rukovanje muzičkim podacima u AWS S3 skladištu


## Opis zadatka
Cilj ovog zadatka je povezati Google Colab okruženje sa AWS S3 skladištem pomoću biblioteke Boto3, učitati sve CSV fajlove iz foldera `raw/`, spojiti ih u jedan DataFrame, izračunati agregirane metrike po izvođaču i formirati top 20 listu izvođača. Dobijeni rezultat se zatim sprema nazad u S3 folder `final/` kao CSV fajl.

## Uvod

U ovom zadatku koristi se AWS S3 kao cloud skladište za muzičke CSV podatke.  
Podaci predstavljaju top liste pjesama sa različitih muzičkih platformi kroz više dana.

Analiza obuhvata sljedeće korake:

1. povezivanje Google Colab okruženja sa AWS S3 servisom
2. pronalaženje i učitavanje svih CSV fajlova iz foldera `raw/`
3. spajanje svih fajlova u jedinstven DataFrame
4. izdvajanje platforme iz naziva fajla
5. agregaciju podataka po izvođaču
6. izračunavanje traženih metrika
7. formiranje top 20 izvođača
8. spremanje rezultata u folder `final/` u S3 bucketu

## Instalacija potrebnih biblioteka

Za rad sa AWS S3 servisom koristi se biblioteka `boto3`, dok se za obradu podataka koristi biblioteka `pandas`.

In [ ]:
!pip install boto3

## Uvoz biblioteka

U ovom koraku uvozimo biblioteke koje će biti potrebne za povezivanje sa AWS S3, rad sa CSV fajlovima i obradu podataka.

In [ ]:
import boto3
import pandas as pd
from io import StringIO

## Povezivanje sa AWS S3 servisom

U ovom dijelu uspostavlja se konekcija sa AWS S3 bucketom pomoću IAM access ključeva.  
Konekcija se ostvaruje pomoću `boto3.client()` funkcije.

In [ ]:
import boto3
import pandas as pd
from io import StringIO

import os

aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = "us-east-1"
BUCKET_NAME = "edita-music-data-2026"

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)

print("S3 konekcija OK")

S3 konekcija OK


## Provjera sadržaja `raw/` foldera

Prije obrade podataka potrebno je provjeriti da li se CSV fajlovi zaista nalaze u folderu `raw/` unutar S3 bucketa.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")

print("Broj pronađenih objekata:", response.get("KeyCount", 0))

for obj in response.get("Contents", [])[:10]:
    print(obj["Key"])

Broj pronađenih objekata: 91
raw/
raw/deezer_top_20_2025-06-01.csv
raw/deezer_top_20_2025-06-02.csv
raw/deezer_top_20_2025-06-03.csv
raw/deezer_top_20_2025-06-04.csv
raw/deezer_top_20_2025-06-05.csv
raw/deezer_top_20_2025-06-06.csv
raw/deezer_top_20_2025-06-07.csv
raw/deezer_top_20_2025-06-08.csv
raw/deezer_top_20_2025-06-09.csv


## Pronalaženje svih CSV fajlova

U ovom koraku izdvajaju se samo fajlovi sa ekstenzijom `.csv` iz `raw/` foldera.  
Na taj način se osigurava da se u analizu uključe samo relevantni podaci.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="raw/")

csv_files = []

for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith(".csv"):
        csv_files.append(key)

print("Ukupan broj CSV fajlova:", len(csv_files))
print("Prvih 5 CSV fajlova:")
for file in csv_files[:5]:
    print(file)

Ukupan broj CSV fajlova: 90
Prvih 5 CSV fajlova:
raw/deezer_top_20_2025-06-01.csv
raw/deezer_top_20_2025-06-02.csv
raw/deezer_top_20_2025-06-03.csv
raw/deezer_top_20_2025-06-04.csv
raw/deezer_top_20_2025-06-05.csv


## Učitavanje CSV fajlova iz AWS S3

Svaki CSV fajl se čita direktno iz S3 bucketa.  
Tokom učitavanja dodaje se i nova kolona `platform`, koja se izdvaja iz naziva fajla.  
Na primjer, iz naziva `spotify_top_20_2025-06-01.csv` izdvaja se vrijednost `spotify`.

Pored toga, dodaje se i kolona `source_file` kako bi se znao izvor svakog reda podataka.

In [ ]:
all_dfs = []

for file_key in csv_files:
    obj = s3.get_object(Bucket=BUCKET_NAME, Key=file_key)
    file_content = obj["Body"].read().decode("utf-8")

    df = pd.read_csv(StringIO(file_content))

    filename = file_key.split("/")[-1]
    platform = filename.split("_")[0].lower()

    df["platform"] = platform
    df["source_file"] = filename

    all_dfs.append(df)

music_df = pd.concat(all_dfs, ignore_index=True)

print("Svi CSV fajlovi su uspješno učitani.")
print("Dimenzije zajedničkog DataFrame-a:", music_df.shape)
music_df.head()

Svi CSV fajlovi su uspješno učitani.
Dimenzije zajedničkog DataFrame-a: (1800, 8)


,position,artist,track,album,duration,date,platform,source_file
0,1,Clairo,Juna,Charm,195,2025-06-01,deezer,deezer_top_20_2025-06-01.csv
1,2,Shaboozey,Good News,Good News,199,2025-06-01,deezer,deezer_top_20_2025-06-01.csv
2,3,Doechii,Anxiety,Anxiety,249,2025-06-01,deezer,deezer_top_20_2025-06-01.csv
3,4,Benson Boone,Beautiful Things,Beautiful Things,180,2025-06-01,deezer,deezer_top_20_2025-06-01.csv
4,5,Morgan Wallen,What I Want,I’m The Problem,184,2025-06-01,deezer,deezer_top_20_2025-06-01.csv


## Pregled strukture podataka

Prije analize potrebno je provjeriti strukturu DataFrame-a, nazive kolona i eventualne nedostajuće vrijednosti.

In [ ]:
print("Nazivi kolona:")
print(music_df.columns.tolist())

print("\nInformacije o DataFrame-u:")
music_df.info()

Nazivi kolona:
['position', 'artist', 'track', 'album', 'duration', 'date', 'platform', 'source_file']

Informacije o DataFrame-u:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   position     1800 non-null   int64 
 1   artist       1800 non-null   object
 2   track        1800 non-null   object
 3   album        1711 non-null   object
 4   duration     1800 non-null   int64 
 5   date         1800 non-null   object
 6   platform     1800 non-null   object
 7   source_file  1800 non-null   object
dtypes: int64(2), object(6)
memory usage: 112.6+ KB


## Provjera nedostajućih vrijednosti

U ovom koraku provjerava se da li postoje prazne vrijednosti u važnim kolonama kao što su:
- `artist`
- `track`
- `position`
- `platform`

In [ ]:
print(music_df.isnull().sum())

position        0
artist          0
track           0
album          89
duration        0
date            0
platform        0
source_file     0
dtype: int64


## Čišćenje podataka

Radi tačne analize uklanjaju se redovi koji nemaju vrijednosti u ključnim kolonama.  
Također se kolona `position` pretvara u numerički tip podataka kako bi se mogle računati agregatne statistike.

In [ ]:
music_df = music_df.dropna(subset=["artist", "track", "position", "platform"])

music_df["position"] = pd.to_numeric(music_df["position"], errors="coerce")
music_df = music_df.dropna(subset=["position"])
music_df["position"] = music_df["position"].astype(int)

print("Podaci su očišćeni.")
print("Nove dimenzije DataFrame-a:", music_df.shape)

Podaci su očišćeni.
Nove dimenzije DataFrame-a: (1800, 8)


## Pregled raspodjele po platformama

Pošto se u zadatku traži i broj platformi na kojima se izvođač pojavljuje, korisno je provjeriti koliko redova dolazi sa svake platforme.

In [ ]:
print(music_df["platform"].value_counts())

platform
deezer     600
itunes     600
spotify    600
Name: count, dtype: int64


## Agregacija podataka po izvođaču

U ovom koraku podaci se grupišu po koloni `artist`, a zatim se računaju tražene metrike:

- `appearances` – ukupan broj pojavljivanja izvođača
- `unique_tracks` – broj različitih pjesama izvođača
- `avg_position` – prosječna pozicija na listama
- `best_position` – najbolja ostvarena pozicija
- `platform_count` – broj različitih platformi na kojima se izvođač pojavio

In [ ]:
artist_ranking = (
    music_df.groupby("artist")
    .agg(
        appearances=("track", "count"),
        unique_tracks=("track", "nunique"),
        avg_position=("position", "mean"),
        best_position=("position", "min"),
        platform_count=("platform", "nunique")
    )
    .reset_index()
)

artist_ranking["avg_position"] = artist_ranking["avg_position"].round(2)

print("Agregacija po izvođaču je završena.")
artist_ranking.head()

Agregacija po izvođaču je završena.


,artist,appearances,unique_tracks,avg_position,best_position,platform_count
0,ATEEZ,12,1,9.58,1,1
1,Alex Warren,56,1,10.96,1,3
2,Alison Goldfrapp,7,1,10.71,5,1
3,Arctic Monkeys,12,1,9.92,2,1
4,Ariana Grande,10,1,13.20,5,1


## Sortiranje izvođača

Nakon agregacije, izvođači se sortiraju tako da prioritet imaju:

1. veći broj pojavljivanja
2. veći broj različitih pjesama
3. veći broj platformi
4. niža prosječna pozicija
5. niža najbolja pozicija

Na taj način dobija se objedinjena rang lista izvođača.

In [ ]:
top_20_artists = artist_ranking.sort_values(
    by=["appearances", "unique_tracks", "platform_count", "avg_position", "best_position"],
    ascending=[False, False, False, True, True]
).head(20).reset_index(drop=True)

top_20_artists.insert(0, "rank", range(1, len(top_20_artists) + 1))

top_20_artists

,rank,artist,appearances,unique_tracks,avg_position,best_position,platform_count
0,1,Sabrina Carpenter,91,4,12.32,1,3
1,2,The Beach Boys,84,7,10.39,1,1
2,3,Teddy Swims,77,3,10.03,1,3
3,4,Benson Boone,75,3,11.45,1,3
4,5,Lady Gaga,59,4,11.97,1,3
5,6,Alex Warren,56,1,10.96,1,3
6,7,Billie Eilish,55,2,9.58,1,3
7,8,Ed Sheeran,49,4,10.82,1,2
8,9,Morgan Wallen,47,4,9.04,1,2
9,10,Chappell Roan,44,2,8.39,1,3


## Top 20 izvođača

Tabela ispod prikazuje 20 najbolje rangiranih izvođača na osnovu agregiranih metrika iz svih učitanih CSV fajlova.

In [ ]:
top_20_artists

,rank,artist,appearances,unique_tracks,avg_position,best_position,platform_count
0,1,Sabrina Carpenter,91,4,12.32,1,3
1,2,The Beach Boys,84,7,10.39,1,1
2,3,Teddy Swims,77,3,10.03,1,3
3,4,Benson Boone,75,3,11.45,1,3
4,5,Lady Gaga,59,4,11.97,1,3
5,6,Alex Warren,56,1,10.96,1,3
6,7,Billie Eilish,55,2,9.58,1,3
7,8,Ed Sheeran,49,4,10.82,1,2
8,9,Morgan Wallen,47,4,9.04,1,2
9,10,Chappell Roan,44,2,8.39,1,3


## Spremanje rezultata u CSV fajl

Dobijena top 20 lista izvođača sprema se lokalno u Colab okruženju kao CSV fajl pod nazivom `top_20_artists.csv`.

In [ ]:
output_file = "top_20_artists.csv"
top_20_artists.to_csv(output_file, index=False, encoding="utf-8")

print(f"Fajl '{output_file}' je uspješno sačuvan.")

Fajl 'top_20_artists.csv' je uspješno sačuvan.


## Upload rezultata u S3 folder `final/`

U završnom koraku rezultat se šalje nazad u AWS S3 bucket, u folder `final/`.  
Na taj način kompletan tok rada ostaje unutar cloud okruženja.

In [ ]:
s3.upload_file(output_file, BUCKET_NAME, "final/top_20_artists.csv")
print("Rezultat je uspješno uploadovan u S3 folder 'final/'.")

Rezultat je uspješno uploadovan u S3 folder 'final/'.


## Provjera sadržaja `final/` foldera

Nakon uploada provjerava se da li se rezultat zaista nalazi u `final/` folderu.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="final/")

print("Sadržaj final foldera:")
for obj in response.get("Contents", []):
    print(obj["Key"])

Sadržaj final foldera:
final/
final/top_20_artists.csv


## Formiranje linka do rezultata

Ako se za fajl `top_20_artists.csv` omogući javni pristup u AWS S3, moguće je pristupiti fajlu preko direktnog URL-a.

In [ ]:
public_url = f"https://{BUCKET_NAME}.s3.amazonaws.com/final/top_20_artists.csv"
print("Javni link do rezultata:")
print(public_url)

Javni link do rezultata:
https://edita-music-data-2026.s3.amazonaws.com/final/top_20_artists.csv


## Zaključak

U ovom zadatku uspješno je ostvarena veza između Google Colab okruženja i AWS S3 skladišta pomoću biblioteke Boto3.  
Svi CSV fajlovi iz foldera `raw/` su učitani i spojeni u jedinstven DataFrame.  
Nakon toga izvršena je agregacija po izvođaču i formirana top 20 lista na osnovu traženih metrika.

Na kraju je rezultat sačuvan kao CSV fajl i uploadovan u folder `final/` unutar S3 bucketa, čime je kompletiran cijeli tok obrade podataka u cloud okruženju.